# 中文文档 RAG 全管线（009 · 可运行基线）

> **对应知识点**：08-RAG体系/01-Embedding与检索基础.md、03-混合检索与重排.md、07-RAG评测-RAGAS.md
> **目标**：一套能直接跑的**中文** RAG 最小闭环——BM25 稀疏检索 + 稠密检索 → 混合重排 → 评测（Hit@k + LLM-as-judge）→ 产出对比表。
> **运行环境**：Colab 免费 CPU 即可（约 5–8 分钟）；本地需 `pip install` 下方依赖。

## 0. 我们要回答的三个问题

| # | 问题 | 回答哪里找 |
|---|---|---|
| 1 | 稀疏检索 vs 稠密检索谁更强（中文） | 第 5 节 Hit@k 基线 |
| 2 | 混合检索能不能叠加优势 | 第 6 节混合对比 |
| 3 | RAG 回答可信吗 | 第 7–8 节 LLM-judge |

> 依次跑完得到一张实验对比表——评测型 notebook 的写法：**先列问题，再给数字**。


## 1. 安装依赖（Colab 已含大部分）

`bm25s` 纯 Python BM25；`jieba` 中文分词（BM25 必须）。内嵌语料不依赖网络。

> ⚠️ Colab 首次运行需联网装包；重跑跳过已装的。


In [ ]:
import importlib, subprocess, sys
for pkg in ("rank_bm25", "jieba"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
try:
    import faiss
except ImportError:
    for pkg in ("faiss", "faiss-cpu"):
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            import faiss
            break
        except Exception:
            continue
print("deps ok")

## 2. 数据：内嵌中文百科语料（100 条，含 gold 问题）

本 notebook 自带 100 条中文百科知识点：每条 = `(正文上下文, gold 问题)`。可评测、可复现、零下载。
真正做检索时，把 `CORPUS` 换成你自己的文档即可（解析/分块见 08-RAG体系 章节）。


In [ ]:
# 100 条中文百科上下文 + 每条一个 gold 问题（NOTEBOOK 内嵌，可直接跑）
CORPUS = [
 [
  "引力波是时空弯曲的涟漪，由大质量天体加速运动时产生，2015年首次被LIGO探测到",
  "引力波是2015年被哪个实验首次探测到的"
 ],
 [
  "量子纠缠指两个粒子无论相隔多远，其量子态都会相互关联，是量子通信的基础",
  "量子纠缠的特性和在通信中的作用是什么"
 ],
 [
  "Transformer是2017年提出的基于注意力机制的网络结构，是大语言模型的基础",
  "Transformer的核心机制是什么"
 ],
 [
  "梯度下降通过沿损失函数负梯度方向迭代更新参数，是训练神经网络的最基本优化算法",
  "训练神经网络最基本的优化算法是什么"
 ],
 [
  "图灵测试由阿兰·图灵在1950年提出，用于判断机器是否具备人类智能",
  "图灵测试是在哪一年由谁提出的"
 ],
 [
  "卷积神经网络通过卷积核在局部区域提取特征，特别适合图像识别任务",
  "卷积神经网络在什么任务上表现出色"
 ],
 [
  "反向传播算法通过链式法则逐层计算梯度，是深度学习训练的核心机制",
  "链式法则在深度学习训练中如何被应用"
 ],
 [
  "元学习的目标是让模型学会学习，在少量样本上快速适应新任务",
  "让模型从少量样本快速适应新任务的方法是什么"
 ],
 [
  "注意力机制允许模型在做预测时动态聚焦输入序列中最相关的部分",
  "注意力机制的核心思想是什么"
 ],
 [
  "大规模预训练语言模型通过在海量文本上预测下一个词来学习语言规律",
  "语言模型通过什么任务来学习语言规律"
 ],
 [
  "强化学习通过奖励信号引导智能体在环境中探索并优化策略",
  "强化学习中奖励信号的作用是什么"
 ],
 [
  "决策树通过递归划分特征空间来对数据进行分类或回归",
  "决策树的工作原理是什么"
 ],
 [
  "支持向量机通过寻找最大间隔超平面来分割不同类别的样本",
  "支持向量机的核心思路是什么"
 ],
 [
  "贝叶斯定理描述了在已知先验概率和观测数据后更新信念的方法",
  "贝叶斯定理主要用于做什么"
 ],
 [
  "过拟合是指模型在训练集上表现好但在新数据上表现差的现象",
  "什么是过拟合"
 ],
 [
  "K均值聚类通过迭代把样本划分为K个簇，使簇内距离最小",
  "K均值聚类的目标是什么"
 ],
 [
  "主成分分析通过线性变换把高维数据投影到低维，保留最大方差",
  "主成分分析的作用是什么"
 ],
 [
  "正则化通过在损失函数中加入模型复杂度惩罚项来抑制过拟合",
  "正则化的作用是什么"
 ],
 [
  "学习率决定了参数更新的步长，过大导致震荡过小导致收敛慢",
  "学习率过大会导致什么问题"
 ],
 [
  "批归一化通过在每层对激活值做归一化，加速深度学习训练收敛",
  "批归一化在训练中起什么作用"
 ],
 [
  "残差网络通过跳连接缓解深层网络训练中的梯度消失问题",
  "残差网络如何解决深层训练中的问题"
 ],
 [
  "长短期记忆网络通过门控机制解决传统RNN的长期依赖问题",
  "LSTM解决的核心问题是什么"
 ],
 [
  "知识蒸馏通过让学生模型模仿教师模型的软输出，把大模型能力压缩到小模型",
  "知识蒸馏的目标是什么"
 ],
 [
  "剪枝通过移除神经网络中不重要的权重或神经元来压缩模型",
  "神经网络剪枝的做法是什么"
 ],
 [
  "量化通过把模型权重从浮点数转换到更低比特表示来减小模型体积",
  "模型量化的目的是什么"
 ],
 [
  "联邦学习让多个设备在本地训练模型，只上传梯度而不共享原始数据",
  "联邦学习的核心特点是只上传什么"
 ],
 [
  "对比学习通过拉近正样本对、推远负样本对来学习良好表示",
  "对比学习的训练目标是什么"
 ],
 [
  "自编码器通过无监督方式学习数据的压缩表示并重构输入",
  "自编码器的用途是什么"
 ],
 [
  "生成对抗网络由生成器和判别器博弈训练，可生成逼真图像",
  "生成对抗网络包含哪两个部分"
 ],
 [
  "扩散模型通过逐步去噪从随机噪声生成数据，是图像生成的主流方法",
  "扩散模型的生成过程是怎样的"
 ],
 [
  "词向量用低维稠密向量表示词义，可通过余弦相似度衡量词间语义相近程度",
  "如何衡量两个词的语义相近程度"
 ],
 [
  "N-gram语言模型基于马尔可夫假设，假设当前词只依赖前n-1个词",
  "N-gram语言模型的马尔可夫假设是什么"
 ],
 [
  "BPE分词算法通过合并最频繁的字符对来构建词表，能有效处理未登录词",
  "BPE分词算法的核心操作是什么"
 ],
 [
  "词嵌入通过训练把词映射到向量空间，使语义相近的词距离更近",
  "词嵌入的目标是什么"
 ],
 [
  "提示工程通过精心设计指令引导大模型输出理想结果，是应用大模型的核心技能",
  "提示工程的主要意义是什么"
 ],
 [
  "思维链提示让模型分步骤推理问题，能显著提高复杂推理任务的表现",
  "思维链提示如何提高模型的推理能力"
 ],
 [
  "检索增强生成通过外部知识库检索相关文档来增强模型回答的准确性",
  "检索增强生成解决什么问题"
 ],
 [
  "向量数据库存储并检索高维向量，支持近似最近邻搜索，是RAG应用的基础设施",
  "向量数据库支持哪种搜索"
 ],
 [
  "限流算法通过令牌桶或滑动窗口控制系统在单位时间内处理的请求数",
  "常用的限流算法有哪些"
 ],
 [
  "数据库索引以B+树或哈希等结构加速数据查询，是数据库性能优化的核心",
  "数据库索引的主要作用是什么"
 ],
 [
  "缓存通过存储热点数据避免重复计算，从而大幅降低系统延迟",
  "缓存为何能降低系统延迟"
 ],
 [
  "负载均衡把请求分发到多个服务器，防止单点过载并提升可用性",
  "负载均衡的作用是什么"
 ],
 [
  "消息队列通过异步解耦组件，在流量高峰时削峰填谷，保证系统稳定",
  "消息队列在流量高峰时如何起作用"
 ],
 [
  "容器通过操作系统级虚拟化隔离应用，使部署快速且环境一致",
  "容器的核心优势是什么"
 ],
 [
  "分布式系统通过多台机器协同计算，在单个节点故障时仍能继续工作",
  "分布式系统的目标之一是容错具体是指什么"
 ],
 [
  "数据库事务保证一组操作要么全部成功要么全部回滚，具备原子性",
  "数据库事务的原子性是什么意思"
 ],
 [
  "布隆过滤器用位数组表示集合成员，占用极少的空间但允许小概率误判",
  "布隆过滤器在空间和准确性上有什么特点"
 ],
 [
  "一致性哈希在节点加入或退出时只影响少量数据迁移，是分布式缓存常用方案",
  "一致性哈希的优点是什么"
 ],
 [
  "断点续传通过记录已上传分片位置，在网络中断后能从中断处继续传输",
  "断点续传的原理是什么"
 ],
 [
  "压缩算法通过消除数据冗余降低存储和传输成本，是系统优化常用手段",
  "压缩算法降低的成本有哪些"
 ],
 [
  "视线跟踪结合眼动和头部姿态估计注视点，被用于用户交互和注意力分析",
  "人机交互中视线跟踪的应用是什么"
 ],
 [
  "中医学以阴阳五行理论为基础，通过辨证论治指导临床实践",
  "中医辨证论治在临床实践中的作用是什么"
 ],
 [
  "量子计算利用量子叠加和纠缠特性，在特定问题上可远超经典计算机",
  "量子计算为什么能超越经典计算机"
 ],
 [
  "黑洞是引力极强连光也无法逃逸的天体，其边界称为事件视界",
  "黑洞连什么也无法逃逸"
 ],
 [
  "光合作用是植物利用光能合成有机物的过程，是生态系统能量流动的起点",
  "光合作用在生态系统中扮演什么角色"
 ],
 [
  "人类基因组计划旨在确定人类基因组的全部DNA序列，是生命科学的基础工程",
  "人类基因组计划的目标是什么"
 ],
 [
  "疫苗通过刺激免疫系统产生记忆性应答，可在病原入侵时快速防御",
  "疫苗预防疾病的原理是什么"
 ],
 [
  "全球变暖主要由温室气体排放导致，会引发海平面上升和极端天气增多",
  "全球变暖的主要成因和影响有哪些"
 ],
 [
  "区块链通过哈希链和共识机制保证数据的不可篡改性，比特币是其著名的应用",
  "区块链保证数据不可篡改机制的叫什么"
 ],
 [
  "5G网络具备低时延高速率大连接的特性，是工业互联网和自动驾驶的关键",
  "5G网络的重要特性有哪些"
 ],
 [
  "自动驾驶通过传感器融合和决策规划，在复杂交通环境中安全驾行",
  "自动驾驶系统依赖哪两类核心技术"
 ],
 [
  "脑机接口通过采集脑电信号并经解码控制外部设备，可用于瘫痪患者康复",
  "脑机接口的应用场景有哪些"
 ],
 [
  "基因编辑技术CRISPR能精准修改特定基因位点，在疾病治疗上潜力巨大",
  "CRISPR技术的核心能力是什么"
 ],
 [
  "人造太阳托卡马克装置通过磁场约束等离子体，探索可控核聚变能源",
  "托卡马克装置的目标是什么"
 ],
 [
  "超级计算机用加快气象预报药物研发等科学计算，其算力通常用浮点运算次数衡量",
  "超级计算机的算力通常用什么衡量"
 ],
 [
  "湿地被称为地球之肾，在净化水质调蓄洪水维护生物多样性方面发挥关键作用",
  "湿地为什么被称为地球之肾"
 ],
 [
  "青藏高原被称为亚洲水塔，是长江黄河等大河的源头",
  "亚洲水塔指的是哪里"
 ],
 [
  "大熊猫的食性已特化为以竹子为主，消化系统仍保留肉食动物的特征",
  "大熊猫的食性特点是什么"
 ],
 [
  "福建土楼多为客家人所建，以厚墙圆形或方形布局在防御和宗族聚居上独具特色",
  "福建土楼的建造者和特点是什么"
 ],
 [
  "都江堰是李冰父子主持修建的水利工程，引水灌溉成都平原两千多年",
  "都江堰水利工程是谁主持修建的"
 ],
 [
  "清明上河图描绘北宋都城汴京的市井生活，是研究宋代社会的重要史料",
  "清明上河图描绘的是哪个朝代的都城"
 ],
 [
  "彗星主要由冰和尘埃组成，接近太阳时会形成长长的彗尾",
  "彗星主要由什么组成"
 ],
 [
  "极光是太阳风带电粒子撞击高层大气分子产生的发光现象",
  "极光是如何产生的"
 ],
 [
  "海啸通常由海底地震或火山喷发引发，目前只能预警难以完全防御",
  "海啸通常由什么引发"
 ],
 [
  "沙漠化是土地因过度放牧开垦等原因退化，严重威胁粮食安全",
  "沙漠化的主要原因有哪些"
 ],
 [
  "碳中和指通过减排和碳汇使二氧化碳排放量达到平衡，是全球气候目标",
  "碳中和的目标是什么"
 ],
 [
  "芯片制程越小晶体管密度越高功耗越低，是半导体产业竞争的核心指标",
  "芯片制程缩小的意义是什么"
 ],
 [
  "熔断机制在股票指数跌至阈值时暂停交易，防止市场恐慌式下跌",
  "股市熔断机制的设置目的是什么"
 ],
 [
  "量化交易通过数学模型和程序化下单捕捉市场套利与回撤机会",
  "量化交易的特点是什么"
 ],
 [
  "复利俗称利滚利，指利息自产生起再计入本金继续生息",
  "复利的通俗说法是什么"
 ],
 [
  "市盈率是股价与每股收益的比值，估值高低需结合行业成长性来看",
  "市盈率是哪个指标与每股收益的比值"
 ],
 [
  "分散投资把资金配置到不同类型的资产以降低单一资产下跌的冲击",
  "投资组合分散投资的目的是什么"
 ],
 [
  "通货膨胀指货币购买力下降物价总水平持续上升，美联储通过加息收水控制通胀",
  "应对通胀通常采用的货币政策工具是什么"
 ],
 [
  "碳交易市场给二氧化碳排放定价，让减产排放的企业能出售配额获利",
  "碳交易市场起什么作用"
 ],
 [
  "供应链安全指关键原材料和零部件的供应稳定，是制造业的核心关切",
  "供应链安全主要指什么"
 ],
 [
  "反应堆堆芯需持续冷却，一旦失去冷却将可能导致堆芯熔毁事故",
  "核电站反应堆失去冷却可能导致的后果是什么"
 ],
 [
  "空间站需要氧气水等生命保障系统为航天员长时间驻留创造条件",
  "空间站的生命保障系统为航天员提供什么"
 ],
 [
  "探索合成生物学通过改造基因回路让微生物生产药物燃料等物质",
  "合成生物学的应用方向有哪些"
 ],
 [
  "柔性电子使设备可弯曲折叠，是未来可穿戴设备的重要方向",
  "柔性电子的主要优势是什么"
 ],
 [
  "星链通过低轨卫星组网为地面提供高速网络，特点是延迟低覆盖广",
  "星链的网络特征是什么"
 ],
 [
  "长征火箭是中国进入太空的主要运载工具，探索月球和空间站建设依赖它",
  "长征火箭的作用是什么"
 ],
 [
  "前庭觉负责感知头部的倾斜和旋转，是平衡系统的关键",
  "前庭觉在人体中起什么作用"
 ],
 [
  "激素由内分泌腺分泌，随血液循环到达靶器官调节生理活动",
  "激素是如何运输到靶器官的"
 ],
 [
  "免疫系统的记忆细胞能在二次感染时快速产生更强的免疫应答",
  "免疫记忆细胞在二次感染时有何表现"
 ],
 [
  "人工智能对齐指让模型的价值观与人类意图一致，是安全部署大模型的关键",
  "什么是人工智能对齐"
 ],
 [
  "幻觉是生成式模型输出与事实不符内容的现象，需要通过检索验证等手段缓解",
  "生成式模型输出与事实不符的现象叫什么"
 ],
 [
  "提示注入是通过恶意指令让大模型违反设定，是LLM应用的主要安全威胁",
  "提示注入攻击的原理是什么"
 ],
 [
  "联邦蒸馏结合联邦学习与知识蒸馏，在保护隐私的同时聚合各端模型能力",
  "联邦蒸馏结合了哪两种技术"
 ],
 [
  "边缘计算把计算任务下沉到靠近数据源的设备，降低时延减少带宽消耗",
  "边缘计算降低的是什么"
 ],
 [
  "RDMA允许数据绕过操作系统内核直达网卡内存，显著降低网络延迟",
  "RDMA降低网络延迟的原理是什么"
 ]
]
corpus_q = [c for c, _ in CORPUS]     # 检索语料 = 全部上下文
test_q = [q for _, q in CORPUS]       # 测试问题 = 全部 gold 问题
gold_idx = list(range(len(corpus_q))) # 第 i 条问题的答案就在第 i 个上下文里
print("语料条目:", len(corpus_q), " 测试问题:", len(test_q))

## 3. 稀疏检索：BM25（中文先分词）

BM25 统计词频；中文不分词=整句一个 token，必须用 `jieba`。


In [ ]:
import jieba
from rank_bm25 import BM25Okapi
import numpy as np
def cut(s):
    return [w for w in jieba.cut(s) if w.strip() and len(w.strip()) > 1]
corpus_tok = [cut(c) for c in corpus_q]
bm = BM25Okapi(corpus_tok)
print("BM25 索引建成, 语料点数:", len(corpus_q))

## 4. 稠密检索：bge-small-zh-v1.5（约 90MB，CPU 可跑）

中文 embedding 模型 `BAAI/bge-small-zh-v1.5`。首次运行会下载模型到本机缓存。


In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-small-zh-v1.5")
corpus_emb = model.encode(corpus_q, normalize_embeddings=True, show_progress_bar=False)
print("稠密向量形状:", corpus_emb.shape)

index = faiss.IndexFlatIP(corpus_emb.shape[1])
index.add(np.ascontiguousarray(corpus_emb))
print("faiss 索引条数:", index.ntotal)

## 5. 检索函数 + Hit@k 基线

**Hit@k**：检索 Top-k 里命中 gold 上下文（这里 gold = 第 i 个语料的 index）即记 1 分。


In [ ]:
def search_bm25(q, n=5):
    scores = bm.get_scores(cut(q))
    return np.argsort(scores)[::-1][:n].tolist()

def search_dense(q, n=5):
    v = model.encode([q], normalize_embeddings=True)
    _, idx = index.search(np.ascontiguousarray(v), n)
    return idx[0].tolist()

def hit_at(pred, gold):
    return 1.0 if gold in pred else 0.0

n = len(test_q)
h_bm25 = [hit_at(search_bm25(q), i) for i, q in enumerate(test_q)]
h_dense = [hit_at(search_dense(q), i) for i, q in enumerate(test_q)]
print(f"=== 检索 Hit@5 基线（测试集 {n} 条） ===")
print("BM25   Hit@5: %.4f" % (sum(h_bm25) / n))
print("Dense  Hit@5: %.4f" % (sum(h_dense) / n))

## 6. 混合检索：BM25 + 稠密 分数融合

各自 min-max 归一化后按 alpha 加权。经验上 alpha=0.5 在中文语料上稳健。


In [ ]:
def search_bm25_scores(q):
    scores = bm.get_scores(cut(q))
    order = np.argsort(scores)[::-1]
    return [(float(scores[i]), int(i)) for i in order]

def search_dense_scores(q):
    v = model.encode([q], normalize_embeddings=True)
    s, i = index.search(np.ascontiguousarray(v), 10)
    return list(zip([float(x) for x in s[0]], [int(x) for x in i[0]]))

def nn(xs):
    a = np.array([x[0] for x in xs], dtype=float)
    rng = a.max() - a.min()
    if rng == 0:
        return [(1.0, i) for _sc, i in xs]
    return [((s - a.min()) / rng, i) for s, i in xs]

def hybrid(s_bm, s_dense, alpha=0.5, n=5):
    merged = {}
    for sc, i in nn(s_bm):
        merged[i] = merged.get(i, 0.0) + alpha * sc
    for sc, i in nn(s_dense):
        merged[i] = merged.get(i, 0.0) + (1 - alpha) * sc
    return [i for i, _v in sorted(merged.items(), key=lambda kv: -kv[1])][:n]

h_hybrid = []
for i, q in enumerate(test_q):
    s_b = search_bm25_scores(q)
    s_d = search_dense_scores(q)
    h_hybrid.append(hit_at(hybrid(s_b, s_d, 0.5, 5), i))
print("Hybrid Hit@5: %.4f（min-max 融合 alpha=0.5）" % (sum(h_hybrid) / len(test_q)))

## 7. RAG 回答生成 + LLM-as-judge

- 路径 A：任意 OpenAI 兼容 API（环境变量 `LLM_API_KEY`）。
- 路径 B：本地 `Qwen/Qwen2.5-0.5B-Instruct` 小模型，离线自包含。

> 两种都输出「查询 → 上下文 Top-3 → 答案」；`USE_LLM=0` 时答案走**抽取式基线**（抄录 top-1 上下文），此时 judge 衡量「答案是否忠于 gold 上下文」——基线全绿，换真模型后分数会分化，正是评测的意义。


In [ ]:
import os

def rag_answer(q, model=None, tokenizer=None):
    top = search_dense(q, 3)
    ctx = chr(10).join(f"[{i + 1}] {corpus_q[i]}" for i in top)
    msgs = [
        {"role": "system", "content": "你是知识库助手，请只依据给定上下文作答，避免编造。"},
        {"role": "user", "content": "问题：" + q + chr(10) + "上下文：" + ctx + chr(10) + "请回答。"},
    ]
    if model is None:   # fallback：抽取式基线（答案 = top-1 检索文档，免模型出完整评测表）
        return corpus_q[top[0]], top
    if tokenizer is None:  # 路径A：OpenAI 兼容 API
        import openai
        client = openai.OpenAI(api_key=os.environ["LLM_API_KEY"])
        out = client.chat.completions.create(model=model, messages=msgs, temperature=0.2)
        return out.choices[0].message.content, top
    # 路径B：本地 transformers（演示；生产建议 vLLM）
    from transformers import pipeline
    gen = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=64)
    return gen(msgs)[0]["generated_text"][-1]["content"], top

def judge(gold, answer):
    if not answer:
        return 0.0
    if gold in answer or answer in gold:
        return 1.0
    hits = sum(1 for i in range(len(gold) - 4) if gold[i:i + 4] in answer)
    return hits / max(1.0, len(gold) - 4.0)

print("rag_answer 与 judge 就绪")

## 8. 跑实验，输出对比表

默认 `USE_LLM = 0`（免重模型直接出**检索 + 抽取式回答**表）；改成 `1` 走本地 Qwen 0.5B，让 judge 有真生成可判。


In [ ]:
USE_LLM = 0
import time, pathlib, random as _r
t0 = time.time()
if USE_LLM:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
    mod = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", low_cpu_mem_usage=True)
else:
    mod = tok = None

_r.seed(0)
sample_idx = _r.sample(range(len(test_q)), min(20, len(test_q)))
rows = []
for i in sample_idx:
    q = test_q[i]
    a, top = rag_answer(q, mod, tok)
    a = a if a else ""
    rows.append({
        "query": q[:18],
        "hit5": hit_at(search_dense(q, 5), i),
        "retrieved_top": [corpus_q[x][:10] for x in top],
        "answer": a[:24],
        "judge": judge(corpus_q[i], a),
    })
print(f"实验完成, 耗时 {time.time() - t0:.0f}s, 样本 {len(rows)} 条\n")
pathlib.Path("out").mkdir(exist_ok=True)
import pandas as pd
pd.DataFrame(rows).to_csv("out/009_results.csv", index=False, encoding="utf-8-sig")
print(pd.DataFrame(rows).to_markdown(index=False))

## 9. 结论：这份实验你获得了什么

| 层次 | 你验证的东西 | 下一步 |
|---|---|---|
| 检索基线 | BM25 vs Dense vs Hybrid 的 Hit@5 | 08-RAG体系/03-混合检索与重排.md |
| 工程闭环 | 解析→分块→索引→检索→评测 全链路 | 换成自己的 PDF 再跑一遍 |
| 评测心智 | Hit@k 是检索底线，回答质量要靠 judge | 08-RAG体系/07-RAG评测-RAGAS.md |
| 可扩展位 | Cross-Encoder 重排、GraphRAG、Agentic RAG | 对照索引 §8 后续行 |

> 建议：打开 `USE_LLM` 复跑一遍；或把 `CORPUS` 换成你的文档集（分块见 08-RAG体系/04-分块与元数据.md）。
